In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import logging
import os
import re
import sys
import zipfile
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import display  # noqa: E402
from scipy.stats import pearsonr, wilcoxon  # noqa: E402

from scripts.analysis_common import participant_label  # noqa: E402
from scripts.notebook_helpers import (  # noqa: E402
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    resolve_notebook_wavelet_cache_dir,
)
from src.analysis import iva_quality  # noqa: E402
from src.analysis.isc import FREQUENCY_BANDS  # noqa: E402
from src.analysis.iva_condition_comparison import (  # noqa: E402
    slice_to_band,
    stack_conditions_on_subject_axis,
)
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    REAL_CONDITIONS,
    ConditionVariants,
    CoordinateSystems,
    ExclusionCategories,
    ExperimentNames,
    IvaComponentArrays,
    IvaVariants,
    MusicTypeVariants,
    PreprocessedDataVariants,
    SingleDataMetadata,
    SpectrumTypeVariants,
)
from src.filtering.dataset_filter import DatasetFilter  # noqa: E402
from src.io.iva_store import list_iva_results, load_iva_components  # noqa: E402
from src.io.loading import assr_electrode_mask  # noqa: E402
from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.visualization.iva_condition_plots import (  # noqa: E402
    plot_condition_mean_tf_maps,
    plot_condition_mean_topomaps,
    plot_participant_condition_tf_maps,
    plot_participant_condition_topomaps,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
%matplotlib inline
print("Setup complete.")

# Analysis of the Stored IVA Components — Both Conditions Concatenated in Time

Reads the components that
[`scripts/run_iva_condition_tracks.py`](../../scripts/run_iva_condition_tracks.py)
wrote with `--store_components` and analyses them. **Nothing is decomposed here** — the
decomposition belongs on a node
([`run_condition_tracks.pbs`](../../jobs/metacentrum/06-iva-condition-comparison/run_condition_tracks.pbs)),
and its figures are lossy summaries of what it recovered.

**The layout this notebook reads.** Placebo and Psilocybin were concatenated along the
**time** axis
([`ConditionVariants.JOINED_TRACKS`](../../src/definitions/fields.py)):

```
participant k  ->  [ ---- Placebo track ---- | ---- Psilocybin track ---- ]
                   0                     T_pl                  T_pl + T_ps
```

Each participant is **one** IVA dataset whose recording spans both conditions, so
`K = P` participants rather than `2P` recordings. What that changes, and it changes a
lot:

- **The components are the same in both conditions by construction.** The mixing was
  estimated once per participant over the whole concatenated recording, so there is no
  component-matching step and no risk of comparing a Placebo component against a
  different Psilocybin one. The contrast is read by slicing the sources back into the
  two time segments.
- **There is one channel topography per (participant, component), shared by both
  conditions.** So there is no topography contrast to draw here — Step 4 emits a
  **single row** rather than two identical rows and an all-zero difference, which would
  read as a null result while really being a statement about the model. For a genuine
  topography contrast use the sibling notebook
  [`iva_component_analysis_joined.ipynb`](iva_component_analysis_joined.ipynb).
- **The conditions keep their own time bases**, so the two segments need not even be
  the same length. The stored file carries the segment order and lengths, so the split
  is recoverable from disk alone.

**Z-scoring happened before the concatenation** (`zscore_mode`, recorded in the file):
with `per_condition` each track was standardised on its own, which normalises an overall
power difference away — so what is compared below is temporal and spectral *structure*,
not amplitude. Step 1 prints which mode the stored run used, because it changes what a
difference in Step 3 or Step 6 means.

**Prerequisite.** A stored entry for the settings in the config cell. If the load
fails, run:

```bash
python scripts/run_iva_condition_tracks.py \
    --experiment assr --n_pca 10 --reuse_wavelets --store_components
```

## Configuration

In [ ]:
# ── Which stored decomposition to analyse ──────────────────────
# These five values are exactly what the store filename encodes, so they have to match
# the run that wrote it. Note that --zscore_mode is NOT part of the filename: a "joint"
# run overwrites the "per_condition" entry of the same --n_pca, so Step 1 prints the
# mode the loaded file actually used.
EXPERIMENT_NAME = ExperimentNames.ASSR
# The time-axis join: one row per participant, both conditions along time.
CONDITION = ConditionVariants.JOINED_TRACKS
VARIANT = IvaVariants.CHANNEL_JOINED_TRACKS
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPE = MusicTypeVariants.ASSR
else:
    MUSIC_TYPE = MusicTypeVariants.CLASSICAL
# The band the run was restricted to, or None for a broadband run.
BAND: str | None = None
# The run's --n_pca, which is also its component count.
N_COMPONENTS_PCA = 5

# Processed-data root the store is resolved against. None = the project's
# data/processed, which is where the CLI writes by default.
STORE_ROOT: Path | None = None

# What resolved the per-participant sign in the run, named on every figure. Matches the
# CLI script.
ALIGNMENT_NOTE = "PC1 of the per-participant TF maps (strongest bin positive)"

# ── Which components to draw ──────────────────────────────────
# None = every stored component. A list is 0-based, matching the IC <k+1> labels.
COMPONENTS_TO_PLOT: list[int] | None = None

# ── Figure output ─────────────────────────────────────────────
SAVE_PLOTS = True
# Per-participant grids are one figure PER COMPONENT and there can be many of them,
# so they are opt-in rather than part of a routine pass through the notebook.
WRITE_PARTICIPANT_GRIDS = False

# ── Reference lines on the TF panels ──────────────────────────
# The ASSR is continuous 40 Hz stimulation, so the stimulation frequency is the row
# worth locating on every map.
TF_FREQ_MARKS: list[float] = [iva_quality.ASSR_FREQ]
MARK_STIMULUS_ONSETS_ON_TF = True

# ── The window the paired contrast summarises ─────────────────
# A TF map is (frequency x time) per component; a paired test needs ONE number per
# (participant, component). This window is that reduction: the mean of the stored,
# sign-aligned source over a frequency band and a time span. Because the sources are
# z-scored along time before the decomposition, a positive mean reads as "more power
# in this band than this recording's own average", not as absolute power.
#
# Default: a narrow band around the ASSR stimulation frequency, whole time axis.
# Set CONTRAST_BAND to a FrequencyBandNames value instead to use a standard band.
CONTRAST_OFFSET = 0.0
CONTRAST_FREQ_RANGE: tuple[float, float] = (
    iva_quality.ASSR_FREQ - CONTRAST_OFFSET,
    iva_quality.ASSR_FREQ + CONTRAST_OFFSET,
)
CONTRAST_BAND: str | None = None  # e.g. "alpha"; overrides CONTRAST_FREQ_RANGE
CONTRAST_TIME_RANGE: tuple[float, float] | None = None  # None = the whole time axis

# ── Stimulus-locked epoch ─────────────────────────────────────
# Taken from src.definitions.constants.AssrEpoch so this notebook cuts the SAME epoch
# as every other onset-locked ASSR analysis: a short pre-onset baseline, then the
# stimulus plus an equally long post-stimulus interval. iva_quality.onset_window caps
# the post-onset span by the shortest inter-onset gap, so an epoch can never reach the
# next stimulus.
MIN_ONSETS_FOR_EPOCH_AVERAGE = 5

# ── An alignment weaker than this is called out ───────────────
# PC1's share of the ensemble power, from the stored sign alignment. Below this there
# is no single dominant shared map, so the group mean of that component — and any
# contrast built on it — is weak evidence.
WEAK_ALIGNMENT_EVR = 0.5

# ── Plots directory ───────────────────────────────────────────
# Canonical notebook layout. The analysis type carries a "_stored" suffix so these
# figures never overwrite the ones the decomposition notebook writes from its own
# in-memory results — same components, different provenance, worth keeping apart.
SPECTRUM_DIR = (
    SpectrumTypeVariants.BROADBAND.value
    if BAND is None
    else SpectrumTypeVariants.BANDS.value
)
ANALYSIS_TYPE = f"{VARIANT.value}_stored"
PLOTS_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR
    / "06-iva-condition-comparison"
    / "plots"
    / EXPERIMENT_NAME.value
    / SPECTRUM_DIR
    / ANALYSIS_TYPE
    / f"pca_{N_COMPONENTS_PCA}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
PLOT_PREFIX = "" if BAND is None else f"{BAND}_"

print(f"Experiment      : {EXPERIMENT_NAME.value}")
print(f"Stored run      : {VARIANT.value} / {CONDITION.value} / {MUSIC_TYPE.value}")
print(f"Spectrum        : {BAND or SpectrumTypeVariants.BROADBAND.value}")
print(f"Components      : {N_COMPONENTS_PCA}  (the run's --n_pca)")
print(f"Store root      : {STORE_ROOT or ProjectPaths.PROCESSED_DATA_DIR}")
print(f"Plots directory : {PLOTS_DIR}  (saving: {SAVE_PLOTS})")

# ── Raw wavelet tracks for the spatial-filter projection ──────
# Read from the notebook SUBSET caches and nothing else. Deliberately not through
# scripts.notebook_helpers.load_paired_condition_wavelets: that calls load_analyzers
# first, which loads the whole concatenated preprocessed recording for the cohort
# before any wavelet is touched, and then reads the source-of-truth wavelet cache —
# ~52 GB per condition for ASSR. Neither finishes in a notebook.
#
# The subset caches are the per-extent copies under the stage-03 notebook, written
# UNCOMPRESSED precisely so they are cheap to read back. Nothing here recomputes a
# wavelet, and nothing reads data/processed/<experiment>/wavelets.
WAVELET_SUBSET_CACHE_DIR: Path = (
    resolve_notebook_wavelet_cache_dir(EXPERIMENT_NAME)
    / SpectrumTypeVariants.BROADBAND.value
)
# Time-axis segment order, so filtered_placebo comes first and filtered_psilocybin
# second.
CONDITIONS_TO_POOL = list(REAL_CONDITIONS)
# The frequency grid the cache was written with; it is part of the cache filename.
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)
# Needed only to read the participant metadata (a CSV parse and a directory listing —
# no EEG is loaded), which is what names the rows of each cache's subject axis.
COORDINATE_SYSTEM = CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# Trim the cached time axis further after loading. None keeps the cached extent.
# The CHANNEL axis is not a knob: the spatial filters are (components x channels) of
# the stored run, so only a cache with exactly that many channels can be used, and the
# selection below refuses anything else rather than silently truncating a filter.
N_TIMES_SUBSET: int | None = None
# Keep only the first N participants of the stored cohort. Applied to the projection,
# so it does lower the peak here.
N_PAIRS_SUBSET: int | None = None

print(f"Wavelet subset cache : {WAVELET_SUBSET_CACHE_DIR}")
print(f"Track trim           : n_times={N_TIMES_SUBSET}, n_pairs={N_PAIRS_SUBSET}")

# ── The binary ASSR-electrode filter (Step 5) ─────────────────
# The include-list in config/assr_electrodes/<coordinate system>.csv — the standard
# fronto-central selection the 40 Hz steady-state response is read from — used as a
# second spatial filter alongside the IVA ones.
#
# True averages over the selected electrodes; False sums them. The mean is the default
# because it keeps the output in the same wavelet-power units as the input and does not
# scale with how many electrodes the list happens to contain, which a raw binary sum
# does.
ASSR_MASK_NORMALIZE = True
# Preprocessing drops the boundary electrodes, so a listed electrode can be absent from
# a recording. True refuses to under-select rather than quietly averaging over fewer
# electrodes than the list names; False accepts the intersection and logs what was
# missing.
ASSR_MASK_STRICT = True

# ── The 40 Hz trial extraction (Steps 6-8) ────────────────────
# The frequency selections the per-trial time courses are read at, as
# (label, centre Hz, half-width Hz). A zero half-width takes the single nearest bin —
# on this 1 Hz grid that IS the 40 Hz row. A positive one averages every bin inside the
# closed interval, which is what survives wavelet smearing and a few Hz of stimulator
# drift. Both are extracted because they answer the same question with different
# exposure to that smearing, and the extracted arrays are small enough that keeping
# both costs nothing.
FREQ_SELECTIONS: list[tuple[str, float, float]] = [
    ("40hz", iva_quality.ASSR_FREQ, 0.0),
    (
        f"{iva_quality.ASSR_FREQ - iva_quality.TF_ANCHOR_HALFWIDTH_HZ:g}"
        f"-{iva_quality.ASSR_FREQ + iva_quality.TF_ANCHOR_HALFWIDTH_HZ:g}hz",
        iva_quality.ASSR_FREQ,
        iva_quality.TF_ANCHOR_HALFWIDTH_HZ,
    ),
]

# Label of the source row the binary ASSR-electrode filter contributes, alongside the
# learned "IC <k>" rows. Both kinds of filter share ONE source axis in Step 6, so a
# trial-level analysis can slice "the same trial under each filter" instead of joining
# two differently-shaped arrays; the label is what tells them apart afterwards.
BINARY_FILTER_LABEL = "ASSR-mask"

# Where the stimulus onsets come from. NOT the stored decomposition — Step 1 prints
# "Onsets stored for: none" for this run — but the sidecar written next to each
# condition's concatenated array. That array is what the wavelet transform was run on,
# so its sample indices address the very time axis the projected tracks are on.
CONCATENATED_DIR: Path = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT_NAME.value
    / PreprocessedDataVariants.CONCATENATED.value
)

# Where the extracted trials are written: under data/processed (gitignored), beside the
# component store the spatial filters came from.
SAVE_TRIALS = True
TRIALS_DIR: Path = (
    (STORE_ROOT or ProjectPaths.PROCESSED_DATA_DIR)
    / EXPERIMENT_NAME.value
    / "assr_trials"
)

print(f"Freq selections : {[name for name, _c, _h in FREQ_SELECTIONS]}")
print(f"Onsets from     : {CONCATENATED_DIR}")
print(f"Trials directory: {TRIALS_DIR}  (saving: {SAVE_TRIALS})")

# ── The participant-level tests (Step 10) ─────────────────────
# Which frequency selection the tests run on. ONE of them, deliberately: testing both
# would double every family below for no new question, since the two are two readings
# of the same response.
TEST_SELECTION = "40hz"

# How a trial's 275-sample time course becomes one number.
#   "stimulus"            mean over the driven interval.
#   "stimulus_minus_rest" that minus the mean over the rest of the epoch, which cancels
#                         whatever offset the per-trial normalisation left behind,
#                         because both halves carry it equally.
# Fix this BEFORE looking at any p-value.
RESPONSE_MEASURE = "stimulus"

# Anchor each participant's component polarity to the ASSR electrodes before testing.
# The run's own sign alignment (PC1 of the TF maps) makes a component's sign consistent
# in the sense PC1 defines, which is NOT "positive means more fronto-central power" —
# measured on this cohort, the sign of a component's pattern weight over the mask
# electrodes splits across participants on every IC. Re-anchoring to the mask makes a
# higher value mean more 40 Hz power over that area for EVERY participant, which is
# what lets the directional prior below be stated at all.
#
# Why this particular anchor is legitimate and a data-driven one is not: this variant
# estimates ONE mixing matrix per participant over the concatenated recording, so the
# pattern is shared by both conditions and cannot favour either. A flip taken from the
# tested quantity instead — e.g. "make Placebo positive" — breaks the exchangeability
# the paired test rests on and inflates the one-sided false-positive rate from 0.05 to
# ~0.68 under a simulated null.
ALIGN_POLARITY_TO_MASK = True

# Test direction for the 4b contrast, computed as Placebo - Psilocybin. The prior is
# that psilocybin LOWERS the 40 Hz response over the fronto-central area, i.e.
# Placebo > Psilocybin, i.e. a positive difference: "greater". This is only meaningful
# because ALIGN_POLARITY_TO_MASK has made "higher = more power there" true of every row.
# One-sided halves the attainable p (floor 1/2**P rather than 2/2**P) and forfeits any
# claim if the effect runs the other way; it is legitimate only because the direction
# was fixed in advance.
#
# 4c stays two-sided regardless — nothing predicts whether a learned component should
# beat a fixed electrode selection.
CONTRAST_ALTERNATIVE = "greater"

print(f"Tests on      : {TEST_SELECTION}, response = {RESPONSE_MEASURE}")

# ── Summary figures (Step 11) ─────────────────────────────────
# One colour per condition, used by every panel so a line never has to be looked up.
CONDITION_COLORS = {
    ConditionVariants.PLACEBO.value: "#0F6E8C",
    ConditionVariants.PSILOCYBIN.value: "#A6357F",
}
# Spread drawn around each mean time course, ACROSS PARTICIPANTS (never across trials —
# trials within a participant are correlated, so their spread understates the real
# uncertainty). "sem" is mean +/- standard error, "iqr" the 25-75 band around the median.
COURSE_SPREAD = "sem"
# Percentile bootstrap over participants for the forest intervals. The Wilcoxon p is
# exact and does not come from this; the interval is only there to show the spread.
N_BOOTSTRAP = 10_000
BOOTSTRAP_SEED = 42
ALPHA = 0.05

## Data Loading — read the stored decomposition

Two calls. `list_iva_results` enumerates what the experiment has on disk, which is the
quickest way to see which runs exist and what settings they used — the filename carries
the variant, the music type, the spectrum and `n_pca`. `load_iva_components` then reads
one entry, either by path or by the run descriptor as here.

In [ ]:
available = list_iva_results(EXPERIMENT_NAME, processed_data_dir=STORE_ROOT)
print(f"Stored IVA results for {EXPERIMENT_NAME.value} ({len(available)}):")
for entry in available:
    print(f"  {entry.parent.name:<14} {entry.name}")
if not available:
    print(
        "  (nothing stored yet — run the CLI with --store_components first:\n"
        f"   python scripts/run_iva_condition_tracks.py --experiment "
        f"{EXPERIMENT_NAME.value} --n_pca {N_COMPONENTS_PCA} --reuse_wavelets "
        "--store_components)"
    )

results = load_iva_components(
    experiment=EXPERIMENT_NAME,
    condition=CONDITION,
    variant=VARIANT,
    music_type=MUSIC_TYPE,
    band=BAND,
    n_pca=N_COMPONENTS_PCA,
    processed_data_dir=STORE_ROOT,
)
print(f"\nLoaded {results.path}")

## Dataset Selection — split the time axis back into the two conditions

The arrays are stored **unsplit**, on the concatenated time axis, which is the model's
own layout. This cell recovers both views:

- **Per condition** (`tf_by_condition`) — `condition_track` slices the stored segment
  out of the time axis, and `times_for` gives that segment's own time base, restarting
  at zero. This is the view for anything that respects each condition's real length.
- **Restacked on a subject axis** (`tf_stacked`) — `stack_conditions_on_subject_axis`
  lays the split out as `(2P, K, F, T_min)` with per-row participant and condition
  labels, which is the layout the shared comparison figures index. Stacking needs one
  common axis, so it trims to the **shorter** segment; the cell reports when that
  actually removes anything.

The channel patterns are **not** split: one mixing matrix per participant covers both
tracks, so there is a single topography per `(participant, component)`.

In [ ]:
LABEL = results.label  # the canonical "JoinedTracks_<MusicType>" product name

tf_concatenated = results.tf_maps  # (P, K, F, T_total) on the concatenated axis
channel_patterns = results.channel_patterns  # (P, K, C) — SHARED by both conditions
freqs = results.freqs  # (F,) Hz
sfreq = results.sfreq

n_participants, n_components, n_freqs, n_total = tf_concatenated.shape
n_channels = results.n_channels

# One row per participant; the conditions live along time, not on this axis.
participants = list(results.participants)
segment_conditions = list(results.segment_conditions)
segment_lengths = list(results.segment_lengths)

# Per-condition view: each segment on its own time base.
tf_by_condition = {
    condition: results.condition_track(IvaComponentArrays.TF_MAP, condition)
    for condition in segment_conditions
}
times_by_condition = {
    condition: results.times_for(condition) for condition in segment_conditions
}

# Subject-axis view, so the shared comparison figures work unchanged.
tf_stacked, subject_participants, subject_conditions = stack_conditions_on_subject_axis(
    tf_by_condition, segment_conditions, participants
)
condition_rows = list(dict.fromkeys(subject_conditions))
n_common = tf_stacked.shape[-1]
times = np.arange(n_common) / sfreq  # the trimmed common axis the stack is on

comp_indices = (
    list(range(n_components))
    if COMPONENTS_TO_PLOT is None
    else list(COMPONENTS_TO_PLOT)
)
out_of_range = [k + 1 for k in comp_indices if not 0 <= k < n_components]
if out_of_range:
    raise ValueError(
        f"COMPONENTS_TO_PLOT names IC {out_of_range}, outside 1..{n_components}."
    )

# Onsets, LOCAL to each condition's segment — the same frame condition_track returns.
onsets_by_condition = {
    condition: results.stimulus_onsets(condition) for condition in segment_conditions
}
reference_onsets = onsets_by_condition.get(segment_conditions[0])
onset_times = (
    reference_onsets[reference_onsets < n_common] / sfreq
    if MARK_STIMULUS_ONSETS_ON_TF and reference_onsets is not None
    else np.array([])
)

# The topomap layout, rebuilt from the stored channel names.
topo_info = results.topo_info()
# The topographies are shared, so their figures get ONE row with this name rather than
# a fabricated per-condition comparison.
SHARED_ROW = " + ".join(segment_conditions) + " (shared)"

print(f"Product      : {LABEL}")
print(
    f"Concatenated : {tf_concatenated.shape}  (participants x components x freqs x times)"
)
print(
    f"Topographies : {channel_patterns.shape}  (participants x components x channels) — shared"
)
print(f"Participants : {n_participants}")
print(f"Segments     : {dict(zip(segment_conditions, segment_lengths))} samples")
for condition, segment_times in times_by_condition.items():
    print(f"  {condition:<12}: {segment_times[-1]:.1f} s @ {sfreq} Hz")
print(f"Stacked view : {tf_stacked.shape}  (trimmed to {n_common} samples)")
if len(set(segment_lengths)) > 1:
    dropped = max(segment_lengths) - n_common
    print(
        f"  NOTE: the segments differ in length, so the stack dropped {dropped} "
        "sample(s) from the longer one. Steps that respect the real lengths use "
        "tf_by_condition instead."
    )
print(f"Components   : showing {len(comp_indices)} of {n_components}")
print(f"Freq axis    : {freqs[0]:.1f}-{freqs[-1]:.1f} Hz ({n_freqs} bins)")

# ── Small helpers used by several steps below ─────────────────


def _grid(n_panels: int, width: float = 3.6, height: float = 2.9):
    """A subplot grid wide enough for *n_panels*, at most 5 columns."""
    ncols = min(5, max(1, n_panels))
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(width * ncols, height * nrows), squeeze=False
    )
    for j in range(n_panels, nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")
    return fig, axes, ncols


def _save(fig, name: str) -> None:
    """Write *fig* into PLOTS_DIR under the band prefix, when saving is on."""
    if not SAVE_PLOTS:
        return
    path = PLOTS_DIR / f"{PLOT_PREFIX}{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved {path}")


def _benjamini_hochberg(pvals) -> np.ndarray:
    """BH step-up adjusted p-values; ``NaN`` entries pass through untouched.

    Written out rather than imported so the notebook has no statsmodels dependency.
    """
    p = np.asarray(pvals, dtype=float)
    adjusted = np.full(p.shape, np.nan)
    finite = ~np.isnan(p)
    if not finite.any():
        return adjusted
    vals = p[finite]
    order = np.argsort(vals)
    n = vals.size
    stepped = vals[order] * n / np.arange(1, n + 1)
    # Enforce monotonicity from the largest p downwards.
    stepped = np.minimum.accumulate(stepped[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(stepped, 0.0, 1.0)
    adjusted[finite] = out
    return adjusted


def _contrast_masks():
    """``(freq_mask, time_mask, description)`` for the paired-contrast window."""
    if CONTRAST_BAND is not None:
        low, high = FREQUENCY_BANDS[CONTRAST_BAND]
        band_name = CONTRAST_BAND
    else:
        low, high = CONTRAST_FREQ_RANGE
        band_name = f"{low:.1f}-{high:.1f} Hz"
    freq_mask = (freqs >= low) & (freqs <= high)
    if not freq_mask.any():
        raise ValueError(
            f"No stored frequency falls in [{low}, {high}] Hz; the stored grid is "
            f"{freqs[0]:.1f}-{freqs[-1]:.1f} Hz. Adjust CONTRAST_FREQ_RANGE / "
            "CONTRAST_BAND, or load a run whose band covers it."
        )
    return freq_mask, band_name


def _time_mask(axis_times: np.ndarray) -> np.ndarray:
    """Boolean mask over *axis_times* for CONTRAST_TIME_RANGE (None = everything)."""
    if CONTRAST_TIME_RANGE is None:
        return np.ones(axis_times.size, dtype=bool)
    start, stop = CONTRAST_TIME_RANGE
    mask = (axis_times >= start) & (axis_times <= stop)
    if not mask.any():
        raise ValueError(
            f"No stored sample falls in [{start}, {stop}] s; the axis spans "
            f"0-{axis_times[-1]:.1f} s. Adjust CONTRAST_TIME_RANGE."
        )
    return mask

---
## Step 1 — What is in the file

Before any analysis, the two things that make a stored decomposition trustworthy:
the **row bookkeeping** (which participant, under which condition, each row of every
array is) and the **axes** (the frequency, time and channel grids the arrays are on).
Both come out of the file; neither is a convention this notebook has to remember.

`participant_frame()` is the mapping in table form — the thing that lets a row index
be turned back into a person.

In [ ]:
print(f"Variant           : {results.variant.value}")
print(f"Product           : {LABEL}   (store condition {results.condition.value})")
print(f"Spectrum          : {results.band or SpectrumTypeVariants.BROADBAND.value}")
print(f"Components        : {n_components}")
print(f"Participants      : {n_participants}  (one row each, both tracks)")
print(f"Concatenated axis : {n_total} samples, {n_total / sfreq:.1f} s @ {sfreq} Hz")
print(f"Segments          : {list(zip(segment_conditions, segment_lengths))}")
print(f"Frequency axis    : {n_freqs} bins, {freqs[0]:.1f}-{freqs[-1]:.1f} Hz")
print(f"Channels          : {n_channels}")
print(f"Stored arrays     : {sorted(results.arrays)}")
print(f"Stored diagnostics: {sorted(results.extras)}")
print(f"Onsets stored for : {results.onset_conditions or 'none'}")

# The z-scoring mode is not in the filename, so read it off the file: it decides
# whether an amplitude difference between the conditions survived at all.
zscore_mode = results.extras.get("zscore_mode")
if zscore_mode is None:
    print("\nz-score mode      : not recorded (a file written before it was stored)")
else:
    mode = zscore_mode.item() if zscore_mode.ndim == 0 else str(zscore_mode)
    print(f"\nz-score mode      : {mode}")
    if mode == "per_condition":
        print(
            "  Each track was standardised on its own, so an overall power difference "
            "between\n  the conditions was normalised away. What the contrasts below "
            "compare is temporal\n  and spectral STRUCTURE, not amplitude."
        )
    else:
        print(
            "  Standardised over the whole concatenated recording, so an overall power "
            "difference\n  between the conditions survives as an offset between the "
            "segments."
        )

# The writer validates these; re-check on read so a hand-edited or truncated file
# cannot quietly mis-address rows.
if sum(segment_lengths) != n_total:
    raise ValueError(
        f"Segments total {sum(segment_lengths)} samples but the stored axis has "
        f"{n_total}."
    )
if len(participants) != n_participants:
    raise ValueError("Row bookkeeping does not match the arrays.")

display(results.participant_frame())

---
## Step 2 — Recover the IVA spatial filters

The file stores the **forward patterns** (the topographies), not the filters, and the
two are not interchangeable. For participant *p*:

- the **filter** `U_p = W_p @ P_p`, shape `(components, channels)`, is the *backward*
  operator — it extracts a source from the channels, which is what is needed to
  project a new signal;
- the stored **pattern** `A_p = pinv(U_p)`, shape `(channels, components)`, is the
  *forward* model — how the source projects *onto* the channels, which is what belongs
  on a topomap.

After `iva_g(..., whiten=True)` the two are nearly unrelated, because the filter
carries a `Σ⁻¹` reweighting that up-weights the lowest-variance retained PCA
directions (Haufe et al., 2014; see
[`iva_component_patterns`](../../src/analysis/wavelet_ica.py)). Using the pattern here
would be the classic filter-vs-pattern error.

The filter is recoverable **exactly**: `U_p` has full row rank, so
`pinv(pinv(U_p)) = U_p`. The cell below inverts the stored patterns and then checks the
round trip rather than trusting it — `U_p @ A_p` must be the identity. The per-participant
sign flips the run applied are diagonal ±1 scalings on the component index, so they
propagate through the inverse and the recovered filters stay consistent with the stored
sources.

**One filter per participant, shared by both tracks.** That is this variant's
construction: the mixing was estimated once over the whole concatenated recording, so
the same `U_p` applies to that participant's Placebo track and to their Psilocybin
track.

In [ ]:
# channel_patterns[p] is (K, C) and holds A_p transposed, so A_p is its transpose.
# Inverting that gives back the (K, C) spatial filter U_p.
spatial_filters = np.stack(
    [np.linalg.pinv(channel_patterns[p].T) for p in range(n_participants)]
)  # (P, K, C)

# Check the round trip instead of assuming it: U_p @ A_p must be the identity. Expect a
# residual around 1e-7, not 1e-15: the store keeps the patterns as float32 by default, so
# the inversion inherits float32 precision. Orders of magnitude above that would mean the
# stored patterns are rank-deficient and the recovered filter is not the operator the run
# actually used.
identity_error = np.array(
    [
        np.abs(spatial_filters[p] @ channel_patterns[p].T - np.eye(n_components)).max()
        for p in range(n_participants)
    ]
)

print(
    f"Spatial filters : {spatial_filters.shape}  (participants x components x channels)"
)
print(
    f"|U_p A_p - I|   : max {identity_error.max():.2e} "
    f"(worst participant {participants[int(identity_error.argmax())]})"
)
if identity_error.max() > 1e-6:
    raise ValueError(
        f"The recovered filters do not invert the stored patterns (max residual "
        f"{identity_error.max():.2e}). The stored patterns are probably "
        "rank-deficient, so the filter cannot be recovered from them."
    )

---
## Step 3 — Read the raw, un-z-scored wavelet tracks from the subset caches

**Caches only.** This step reads the per-extent subset caches under
`notebooks/03-wavelet-analysis/wavelet_cache/` and nothing else: no wavelet is
recomputed, the 52 GB source-of-truth caches under
`data/processed/<experiment>/wavelets/` are never opened, and no preprocessed
recording is loaded.

That is a change from the obvious route.
`load_paired_condition_wavelets` would call `load_analyzers` first, which loads the
whole concatenated recording for the cohort *before* touching a wavelet, and then reads
the source cache — which is why it never finished. The subset caches are written
uncompressed for exactly this purpose, and they already carry everything needed to
interpret them: the array as `(subjects, channels, frequencies, times)`, the channel
names, the frequency grid and the sampling rate.

**No standardisation at any stage.** The cache holds wavelet power as it was
transformed; nothing here z-scores it, per track or jointly. The output of Step 4 is
therefore in the cache's own power units, so an overall power difference between the
conditions survives into the result instead of being normalised away. (The *filter* was
estimated on z-scored data — that is baked into the stored decomposition and cannot be
undone here. What changes is the signal it is applied to.)

**The one thing that has to match is the channel axis.** The filters are
`(components x channels)` of the stored run, so a cache with a different channel count
is unusable — dropping columns from a spatial filter does not restrict it to those
channels, it makes a different and meaningless operator. The selection below therefore
requires an exact channel-count match per condition and reports precisely what to build
if it is missing.

Participant labels come from the dataset **metadata** (a CSV parse plus a directory
listing), because a cache's subject axis is in concatenation order and carries no
labels of its own.

In [ ]:
def _npz_headers(path: Path) -> dict[str, tuple]:
    """Array shapes and dtypes inside an ``.npz``, WITHOUT decompressing anything.

    Lets the inventory below report what a cache holds at no cost; reading ``data``
    normally would pull the whole tensor into memory.
    """
    readers = {
        (1, 0): np.lib.format.read_array_header_1_0,
        (2, 0): np.lib.format.read_array_header_2_0,
    }
    out: dict[str, tuple] = {}
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            if not name.endswith(".npy"):
                continue
            with archive.open(name) as handle:
                version = np.lib.format.read_magic(handle)
                shape, _fortran, dtype = readers[version](handle)
            out[name[: -len(".npy")]] = (shape, str(dtype))
    return out


def _subset_cache_entries(condition) -> list[dict]:
    """Subset-cache entries for one condition, at this frequency grid.

    Filenames are ``<label>__wavelet_<repr>__<f0>_<f1>_<n>__S<n>_C<c>_T<t>__freqdim1r``,
    written by ``scripts.notebook_helpers._save_subset_cache``. Parsed rather than
    reconstructed, because the extent is not known in advance — that is exactly what the
    inventory is for.
    """
    label = f"{condition.value}_{MUSIC_TYPE.value}"
    freq_signature = f"{FREQS[0]:.3f}_{FREQS[-1]:.3f}_{len(FREQS)}"
    pattern = f"{label}__wavelet_power__{freq_signature}__*__freqdim1r.npz"
    entries = []
    for candidate in sorted(WAVELET_SUBSET_CACHE_DIR.glob(pattern)):
        match = re.search(r"__S(\d+)_C(\d+)_T(\d+)__", candidate.name)
        if match is None:
            continue  # a "full"-extent entry; not usable without knowing its shape
        shape, dtype = _npz_headers(candidate).get("data", (None, None))
        entries.append(
            {
                "condition": condition.value,
                "path": candidate,
                "n_subjects": int(match.group(1)),
                "n_channels": int(match.group(2)),
                "n_times": int(match.group(3)),
                "shape": shape,
                "dtype": dtype,
                "size_gb": candidate.stat().st_size / 1e9,
            }
        )
    return entries


inventory = [
    entry
    for condition in CONDITIONS_TO_POOL
    for entry in _subset_cache_entries(condition)
]
print(f"Subset caches in {WAVELET_SUBSET_CACHE_DIR}:")
if not inventory:
    print("  (none for this experiment / frequency grid)")
for entry in inventory:
    usable = "USABLE" if entry["n_channels"] == n_channels else f"needs C{n_channels}"
    print(
        f"  {entry['condition']:<12} S{entry['n_subjects']:<3} "
        f"C{entry['n_channels']:<4} T{entry['n_times']:<6} "
        f"{entry['size_gb']:6.2f} GB  {entry['dtype']}  [{usable}]"
    )

# ── Pick one entry per condition: the channel count must match the run ─
selected = {}
for condition in CONDITIONS_TO_POOL:
    candidates = [
        entry
        for entry in _subset_cache_entries(condition)
        if entry["n_channels"] == n_channels
    ]
    if not candidates:
        available = [
            f"C{entry['n_channels']}_T{entry['n_times']}"
            for entry in _subset_cache_entries(condition)
        ]
        # Both flags matter: the subset cache is keyed by channels AND time, so
        # omitting --n_times writes a "full"-extent entry instead. Suggest the time
        # extent an already-usable entry uses, so the two conditions end up matched.
        usable_times = [
            entry["n_times"]
            for other in CONDITIONS_TO_POOL
            for entry in _subset_cache_entries(other)
            if entry["n_channels"] == n_channels
        ]
        suggested_times = (
            max(usable_times) if usable_times else (N_TIMES_SUBSET or 3000)
        )
        raise FileNotFoundError(
            f"No subset cache for {condition.value}_{MUSIC_TYPE.value} with "
            f"{n_channels} channel(s), which is what the stored run used. A cache with "
            "a different channel count cannot be substituted: the spatial filters are "
            "indexed by those channels.\n"
            "Build it once (it reads the source cache, so run it as a job, not here):\n"
            f"  python scripts/run_iva_condition_tracks.py --experiment "
            f"{EXPERIMENT_NAME.value} --n_pca {N_COMPONENTS_PCA} --n_channels "
            f"{n_channels} --n_times {suggested_times} --reuse_wavelets "
            f"--subset_cache\n"
            f"Available for this condition: {available or 'nothing'}"
        )
    # Longest time axis available, then trimmed by N_TIMES_SUBSET below.
    selected[condition] = max(candidates, key=lambda entry: entry["n_times"])
    print(f"\nUsing for {condition.value}: {selected[condition]['path'].name}")

# ── Participant labels for each cache's subject axis (metadata only) ───
handler = DatasetHandler(EXPERIMENT_NAME, COORDINATE_SYSTEM)
raw_participants = {}
for condition in CONDITIONS_TO_POOL:
    frame = DatasetFilter.filter_dataset_by_all_categories(
        handler.dataset_metadata,
        handler.excluded_participants_metadata,
        [MUSIC_TYPE],
        [condition],
        EXCLUSION_CATEGORIES,
    )
    raw_participants[condition.value] = [
        participant_label(pid) for pid in frame[SingleDataMetadata.PARTICIPANT_ID]
    ]

# ── Read the arrays ───────────────────────────────────────────────────
raw_tracks = {}
raw_freqs = FREQS
for condition in CONDITIONS_TO_POOL:
    entry = selected[condition]
    with np.load(entry["path"]) as cached:
        track = cached["data"]  # (S, C, F, T) — already reshaped by the writer
        cache_freqs = cached["freqs"]
        cache_sfreq = float(cached["sfreq"])
        cache_channels = (
            cached["feature_names"].tolist()
            if cached["has_feature_names"].item()
            else None
        )
    if N_TIMES_SUBSET is not None:
        track = track[..., :N_TIMES_SUBSET]

    labels = raw_participants[condition.value]
    if len(labels) != track.shape[0]:
        raise ValueError(
            f"{condition.value}: the cache has {track.shape[0]} subject(s) but the "
            f"metadata names {len(labels)}. The cache was written for a different "
            "cohort or exclusion set."
        )
    if cache_channels is not None and cache_channels != list(results.channel_names):
        raise ValueError(
            f"{condition.value}: the cache's channel axis does not match the stored "
            f"run's.\n  cache: {cache_channels[:5]}...\n"
            f"  store: {list(results.channel_names)[:5]}...\n"
            "The spatial filters are indexed by the stored channels."
        )
    if cache_sfreq != sfreq:
        raise ValueError(
            f"{condition.value}: cache sfreq {cache_sfreq} != stored {sfreq}."
        )
    if BAND is not None:
        track, raw_freqs = slice_to_band(track, cache_freqs, BAND)
    elif not np.allclose(cache_freqs, freqs):
        raise ValueError(
            f"{condition.value}: the cache spans "
            f"{cache_freqs[0]:.1f}-{cache_freqs[-1]:.1f} Hz ({cache_freqs.size} bins) "
            f"but the stored run has {freqs[0]:.1f}-{freqs[-1]:.1f} Hz "
            f"({n_freqs} bins)."
        )
    raw_tracks[condition.value] = track

print("\nRaw tracks (un-z-scored), from the subset caches:")
for name, track in raw_tracks.items():
    print(
        f"  {name:<12}: {track.shape}  (subjects x channels x freqs x times), "
        f"{track.nbytes / 1e9:.2f} GB"
    )
print(f"Channel axis  : {n_channels} channel(s), matching the stored run")
print("z-scoring     : none applied at any stage")

---
## Step 4 — Project each track through its participant's filter

The projection itself. For every participant, their filter `U_p` is applied to their
**Placebo** track and, separately, to their **Psilocybin** track:

```
sources_p(f, t) = U_p @ X_p(:, f, t)        # (K, C) @ (C,) -> (K,)
```

Because the filter only contracts the channel axis, frequency and time pass through
untouched, and the two tracks keep their own lengths — no trimming to a common axis,
no concatenation.

Rows are matched **by participant label**, not by position: the store's row order and
the cache's need not agree, and a positional match that happens to work on one cohort
would silently mis-assign filters on another.

The result is the two filtered wavelet signals, in order:

1. `filtered_placebo` — `(participants, components, frequencies, times_placebo)`
2. `filtered_psilocybin` — `(participants, components, frequencies, times_psilocybin)`

In [ ]:
# Match by participant label, never by position: the store's row order and each
# cache's subject axis are independent, and the two conditions' cohorts differ in size.
filter_row = {participant: i for i, participant in enumerate(participants)}
cache_row = {
    name: {participant: i for i, participant in enumerate(labels)}
    for name, labels in raw_participants.items()
}

# Only participants the store has a filter for AND both caches contain.
projected_participants = [
    participant
    for participant in participants
    if all(participant in cache_row[c.value] for c in CONDITIONS_TO_POOL)
]
dropped = [p for p in participants if p not in projected_participants]
if not projected_participants:
    raise ValueError(
        "No stored participant appears in both caches; there is nothing to project."
    )
if dropped:
    print(f"Not in both caches, dropped: {dropped}")
if N_PAIRS_SUBSET is not None:
    projected_participants = projected_participants[:N_PAIRS_SUBSET]
    print(f"Trimmed to {len(projected_participants)} participant(s)")

filtered_by_condition = {}
for condition in CONDITIONS_TO_POOL:
    track = raw_tracks[condition.value]  # (S_c, C, F, T_c)
    projected = np.empty(
        (len(projected_participants), n_components) + track.shape[2:],
        dtype=track.dtype,
    )
    for i, participant in enumerate(projected_participants):
        # (K, C) x (C, F, T) -> (K, F, T): contracts channels only, so frequency and
        # time pass through untouched and each track keeps its own length.
        projected[i] = np.tensordot(
            spatial_filters[filter_row[participant]],
            track[cache_row[condition.value][participant]],
            axes=([1], [0]),
        )
    filtered_by_condition[condition.value] = projected

# The two outputs, in the segment order the run used.
filtered_placebo = filtered_by_condition[CONDITIONS_TO_POOL[0].value]
filtered_psilocybin = filtered_by_condition[CONDITIONS_TO_POOL[1].value]

print(f"Projected participants : {projected_participants}")
print(
    f"filtered_placebo       : {filtered_placebo.shape}  "
    f"(participants x components x freqs x times)"
)
print(f"filtered_psilocybin    : {filtered_psilocybin.shape}")
print(
    f"Value range            : placebo [{filtered_placebo.min():.3g}, "
    f"{filtered_placebo.max():.3g}], psilocybin "
    f"[{filtered_psilocybin.min():.3g}, {filtered_psilocybin.max():.3g}]"
)
print("Units                  : wavelet power, un-z-scored")

---
## Step 5 — Extract the TF maps with the binary ASSR-electrode filter

A second spatial filter, used exactly where the IVA ones are used. It comes from the
checked-in include-list in
[`config/assr_electrodes/<coordinate system>.csv`](../../config/assr_electrodes) — the
standard fronto-central selection the 40 Hz steady-state response is read from — turned
into a `(channels,)` 0/1 vector by
[`assr_electrode_mask`](../../src/io/loading.py), aligned to the stored run's channel
axis.

It is the same *kind* of operator as an IVA filter: a weighting over channels that
contracts the channel axis and leaves frequency and time untouched. The differences are
what make it worth having alongside:

- **Fixed, not learned.** The weights come from the paradigm and the montage, so they
  are identical for every participant and every condition — no per-participant
  estimation, nothing to sign-align.
- **One variant.** Where IVA gives `K` components, this gives a single map, so the
  output is one `(frequencies, times)` TF map per participant per condition rather than
  `K` of them.

Rows are built from the same participant list and the same cache bookkeeping as Step 4,
so `assr_filtered_placebo[i]` and `filtered_placebo[i]` are the same participant's
Placebo track under the two filters, directly comparable.

In [ ]:
# The 0/1 channel selection, aligned to the stored run's channel axis.
assr_mask = assr_electrode_mask(
    list(results.channel_names), COORDINATE_SYSTEM, strict=ASSR_MASK_STRICT
)
assr_channels = [name for name, keep in zip(results.channel_names, assr_mask) if keep]

# One filter row: the K = 1 analogue of spatial_filters, so the projection below is the
# same contraction Step 4 does.
assr_filter = assr_mask.astype(float)
if ASSR_MASK_NORMALIZE:
    assr_filter = assr_filter / assr_filter.sum()

print(
    f"ASSR electrodes : {int(assr_mask.sum())} of {assr_mask.size} channel(s) "
    f"({'mean' if ASSR_MASK_NORMALIZE else 'sum'} over them)"
)
print(f"                  {assr_channels}")

assr_by_condition = {}
for condition in CONDITIONS_TO_POOL:
    track = raw_tracks[condition.value]  # (S_c, C, F, T_c)
    projected = np.empty(
        (len(projected_participants),) + track.shape[2:], dtype=track.dtype
    )
    for i, participant in enumerate(projected_participants):
        # (C,) x (C, F, T) -> (F, T): contracts channels only, exactly as the IVA
        # filters do, so the frequency and time axes are the track's own.
        projected[i] = np.tensordot(
            assr_filter,
            track[cache_row[condition.value][participant]],
            axes=([0], [0]),
        )
    assr_by_condition[condition.value] = projected

# The two masked TF-map sets, in the same order as Step 4's outputs.
assr_filtered_placebo = assr_by_condition[CONDITIONS_TO_POOL[0].value]
assr_filtered_psilocybin = assr_by_condition[CONDITIONS_TO_POOL[1].value]

print(f"\nProjected participants     : {projected_participants}")
print(
    f"assr_filtered_placebo      : {assr_filtered_placebo.shape}  "
    f"(participants x freqs x times)"
)
print(f"assr_filtered_psilocybin   : {assr_filtered_psilocybin.shape}")
print(
    f"Value range                : placebo [{assr_filtered_placebo.min():.3g}, "
    f"{assr_filtered_placebo.max():.3g}], psilocybin "
    f"[{assr_filtered_psilocybin.min():.3g}, {assr_filtered_psilocybin.max():.3g}]"
)
print("Units                      : wavelet power, un-z-scored")

---
## Step 6 — Put both filters on one source axis and take the 40 Hz band

Steps 4 and 5 produced two arrays of different rank — `(P, K, F, T)` for the learned
IVA components and `(P, F, T)` for the fixed binary mask — because one filter has `K`
rows and the other has one. That difference is an accident of how many rows each
operator happens to have, not a difference in kind: both are a weighting over channels
that contracts the channel axis and leaves frequency and time untouched. So they are
concatenated here into a **single source axis**

```
sources: (participants, K + 1, frequencies, times)
         ^                ^
         |                +-- IC 1 .. IC K, then "ASSR-mask"
         +-- the same participant order as Steps 4-5
```

which makes "the same participant's same trial under each filter" a slice rather than
a join. `SOURCE_LABELS` is what distinguishes them afterwards, and the mask row stays
last so `sources[:, :n_components]` is still exactly the learned set.

**The 40 Hz band, two ways.** The frequency axis is then collapsed, once per entry of
`FREQ_SELECTIONS`:

- **`40hz`** — the single nearest bin. The cache grid is 1 Hz spaced from 1 to 50 Hz,
  so this is the exact 40 Hz row, with no neighbouring frequency mixed in.
- **`35-45hz`** — the mean over every bin inside ±`TF_ANCHOR_HALFWIDTH_HZ`. Wider than
  the response, deliberately: it is insensitive to wavelet smearing and to a few Hz of
  stimulator drift, at the cost of admitting some off-frequency power.

Neither is obviously right, they are cheap, and which one a result survives under is
itself informative — so both are carried through to the trials.

**Still un-z-scored.** Nothing here standardises or baselines anything; the values stay
in the cache's own wavelet-power units.


In [ ]:
# ── One source axis for both kinds of spatial filter ──────────
# The learned components first, in component order, then the single fixed mask row.
SOURCE_LABELS = [f"IC {k + 1}" for k in range(n_components)] + [BINARY_FILTER_LABEL]

sources_by_condition = {
    condition.value: np.concatenate(
        [
            filtered_by_condition[condition.value],  # (P, K, F, T)
            assr_by_condition[condition.value][:, None],  # (P, 1, F, T)
        ],
        axis=1,
    )
    for condition in CONDITIONS_TO_POOL
}

n_sources = len(SOURCE_LABELS)
for condition in CONDITIONS_TO_POOL:
    stacked = sources_by_condition[condition.value]
    if stacked.shape[1] != n_sources:
        raise ValueError(
            f"{condition.value}: stacked {stacked.shape[1]} source(s) but there are "
            f"{n_sources} labels."
        )


# ── Collapse the frequency axis, once per selection ───────────
def _selection_bins(center: float, halfwidth: float) -> np.ndarray:
    """Bin indices of ``center +/- halfwidth`` on the cache's frequency grid.

    A zero half-width is the single nearest bin. A positive one takes every bin in the
    closed interval, and falls back to the nearest bin if the interval happens to fall
    between two — an empty selection would silently produce a NaN track.
    """
    if halfwidth < 0:
        raise ValueError(f"Half-width must be >= 0; got {halfwidth}.")
    if halfwidth == 0:
        return np.array([int(np.argmin(np.abs(raw_freqs - center)))])
    inside = np.flatnonzero(
        (raw_freqs >= center - halfwidth) & (raw_freqs <= center + halfwidth)
    )
    return inside if inside.size else np.array([int(np.argmin(np.abs(raw_freqs - center)))])


band_tracks: dict[str, dict[str, np.ndarray]] = {}  # selection -> condition -> (P,S,T)
selection_bins: dict[str, np.ndarray] = {}

for name, center, halfwidth in FREQ_SELECTIONS:
    bins = _selection_bins(center, halfwidth)
    selection_bins[name] = bins
    band_tracks[name] = {
        # Mean over the selected bins; a single-bin selection is that bin verbatim.
        condition.value: sources_by_condition[condition.value][:, :, bins, :].mean(
            axis=2
        )
        for condition in CONDITIONS_TO_POOL
    }

print(f"Sources        : {n_sources}  {SOURCE_LABELS}")
print(
    f"Stacked        : "
    f"{ {c.value: sources_by_condition[c.value].shape for c in CONDITIONS_TO_POOL} }"
)
print(f"Frequency grid : {raw_freqs[0]:.1f}-{raw_freqs[-1]:.1f} Hz ({raw_freqs.size} bins)")
print("\n40 Hz band tracks (participants x sources x times), un-z-scored:")
for name, _center, _halfwidth in FREQ_SELECTIONS:
    bins = selection_bins[name]
    shapes = {c.value: band_tracks[name][c.value].shape for c in CONDITIONS_TO_POOL}
    print(
        f"  {name:<9}: bins {bins.tolist()} = {raw_freqs[bins].tolist()} Hz -> {shapes}"
    )

---
## Step 7 — Cut the 40 Hz tracks into stimulus-locked trials

The step that turns a continuous 40 Hz time course into the unit a trial-level
analysis works with: one fixed window per stimulus onset, **kept separately** rather
than averaged. `iva_quality.epoch_average` already exists for the averaging case; this
is deliberately the other one.

**Where the onsets come from.** Not from the stored decomposition — Step 1 reported
`Onsets stored for: none`, because that run predates onsets being written into the
component store. They come from
`data/processed/<experiment>/concatenated/<Condition>_ASSR.stimulus_onsets.npy`, the
sidecar of the concatenated array. That array is the one the wavelet transform was run
on, so its sample indices address exactly the time axis the projected tracks are on —
no shifting, no rescaling.

**The window comes from the paradigm, not from the recording.**
[`AssrEpoch`](../../src/definitions/constants.py) sets a `PRE_ONSET_S` baseline and a
`POST_ONSET_S` span (stimulus plus an equally long tail), and the recording only
*caps* the latter: `iva_quality.onset_window` shortens `post` by the shortest observed
inter-onset gap so an epoch can never reach the next stimulus. Deriving the length
from the gap alone would make the window a property of whatever jitter this recording
happened to have. The two conditions then share the **shorter** `post` of the two, so
a Placebo trial and a Psilocybin trial are the same number of samples and can be
contrasted sample for sample.

**The 12 s cache is the binding constraint here.** The full recording is 188 s and
carries 148 onsets per condition; the subset cache these tracks came from is 3000
samples (12 s), so only the onsets whose whole window fits inside it survive. The cell
prints how many that is against how many exist, because it is the single most
important caveat on everything downstream — extending it means re-projecting from a
longer cache, not re-cutting these arrays.

**No baseline subtraction.** The pre-onset samples are kept as they are, so a baseline
correction remains a choice the analysis can make (or not) rather than one already
baked in.


In [ ]:
# ── The onsets, from the concatenated array's sidecar ─────────
sidecar_onsets = {}
for condition in CONDITIONS_TO_POOL:
    onsets_path = CONCATENATED_DIR / (
        f"{condition.value}_{MUSIC_TYPE.value}{ProjectPaths.STIMULUS_ONSETS_SUFFIX}"
    )
    if not onsets_path.exists():
        raise FileNotFoundError(
            f"No stimulus onsets for {condition.value} at {onsets_path}. They are "
            "written next to the concatenated array by the stimulus alignment; "
            "without them there are no trials to cut."
        )
    sidecar_onsets[condition.value] = np.load(onsets_path).astype(int)

# ── One epoch window, shared by both conditions ───────────────
# Each condition proposes its own (pre, post) from its own onsets; the shared window is
# the shorter post, so the two conditions' trials are the same length and comparable
# sample for sample.
geometry = {}
for condition in CONDITIONS_TO_POOL:
    track_times = raw_tracks[condition.value].shape[-1]
    onsets = sidecar_onsets[condition.value]
    inside = onsets[(onsets >= 0) & (onsets < track_times)]
    if inside.size == 0:
        raise ValueError(
            f"{condition.value}: none of the {onsets.size} onset(s) falls inside the "
            f"{track_times}-sample cached track."
        )
    pre, post = iva_quality.onset_window(inside, track_times, sfreq)
    geometry[condition.value] = (inside, pre, post, track_times)

epoch_pre = {pre for _o, pre, _post, _n in geometry.values()}
if len(epoch_pre) > 1:
    raise ValueError(
        f"Conditions disagree on the pre-onset baseline ({sorted(epoch_pre)} samples); "
        "they cannot share an epoch window."
    )
EPOCH_PRE = epoch_pre.pop()
EPOCH_POST = min(post for _o, _pre, post, _n in geometry.values())
epoch_times = np.arange(-EPOCH_PRE, EPOCH_POST) / sfreq
stimulus_mask = AssrEpoch.stimulus_mask(epoch_times)  # the driven interval


def _cut_trials(array: np.ndarray, onsets: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """One window per onset from the LAST axis, every trial kept.

    The per-trial counterpart of ``iva_quality.epoch_average``, which collapses the
    same windows to their mean. Windows overhanging either end are dropped, and the
    onsets that survived come back with the data so a trial index is never guessed.
    Returns ``(..., n_trials, EPOCH_PRE + EPOCH_POST)`` — the trial axis sits just
    before time.
    """
    n_times = array.shape[-1]
    kept = np.asarray(
        [
            int(onset)
            for onset in onsets
            if int(onset) - EPOCH_PRE >= 0 and int(onset) + EPOCH_POST <= n_times
        ],
        dtype=int,
    )
    if kept.size == 0:
        raise ValueError(
            f"No {EPOCH_PRE + EPOCH_POST}-sample window fits inside the "
            f"{n_times}-sample track."
        )
    windows = [array[..., o - EPOCH_PRE : o + EPOCH_POST] for o in kept]
    return np.stack(windows, axis=-2), kept


# ── Cut every selection, for every condition ──────────────────
trials: dict[str, dict[str, np.ndarray]] = {}  # selection -> condition -> (P,S,N,W)
trial_onsets: dict[str, np.ndarray] = {}  # condition -> (N,), the kept onsets

for name, _center, _halfwidth in FREQ_SELECTIONS:
    trials[name] = {}
    for condition in CONDITIONS_TO_POOL:
        cut, kept = _cut_trials(
            band_tracks[name][condition.value], geometry[condition.value][0]
        )
        trials[name][condition.value] = cut
        trial_onsets[condition.value] = kept  # identical across selections

print(
    f"Epoch window : {EPOCH_PRE} pre + {EPOCH_POST} post = "
    f"{EPOCH_PRE + EPOCH_POST} samples = "
    f"[{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s @ {sfreq} Hz"
)
print(
    f"               stimulus 0-{AssrEpoch.STIMULUS_DURATION_S:.2f} s = "
    f"{int(stimulus_mask.sum())} sample(s) of the window"
)
print("\nOnsets available vs usable, per condition:")
for condition in CONDITIONS_TO_POOL:
    name = condition.value
    inside, _pre, post, track_times = geometry[name]
    total = sidecar_onsets[name].size
    kept = trial_onsets[name]
    print(
        f"  {name:<12}: {total} in the recording, {inside.size} inside the cached "
        f"{track_times} samples ({track_times / sfreq:.1f} s), {kept.size} whole "
        f"window(s) kept"
    )
    if post < AssrEpoch.post_onset_samples(sfreq):
        print(
            f"                post trimmed to {post} samples "
            f"({post / sfreq:.3f} s) by the shortest inter-onset gap"
        )
if any(trial_onsets[c.value].size < MIN_ONSETS_FOR_EPOCH_AVERAGE for c in CONDITIONS_TO_POOL):
    print(
        f"\n  WARNING: fewer than MIN_ONSETS_FOR_EPOCH_AVERAGE "
        f"({MIN_ONSETS_FOR_EPOCH_AVERAGE}) trials in at least one condition."
    )

print("\nTrials (participants x sources x trials x samples), un-z-scored, no baseline:")
for name, _center, _halfwidth in FREQ_SELECTIONS:
    shapes = {c.value: trials[name][c.value].shape for c in CONDITIONS_TO_POOL}
    print(f"  {name:<9}: {shapes}")

---
## Step 8 — Normalise every trial by its own pre-stimulus interval

Each trial is referenced to its **own** `times < 0` window (25 samples, −0.100 to
−0.004 s) rather than to a condition- or participant-level average. That is what makes
this worth doing: the level of 40 Hz power drifts across the recording and differs by
participant, and a per-trial baseline removes both without any group statistic entering
the correction.

**Two forms, because one form does not fit both kinds of source.** The diagnostic that
decides this is in the cell below, and it is not a detail:

| source | per-trial baselines that are negative | worst \|x / baseline\| |
|---|---|---|
| `IC 1`–`IC 5` | 20–56 of 108 per condition | 27 … 3790 |
| `ASSR-mask` | 0 of 108 | 8.5 / 23.4 |

The mask is a non-negative average over electrodes, so dividing by its baseline is
exactly the conventional relative-power change. An IVA source is a **signed**
combination of channels: its power runs negative on roughly half the trials, and a few
baselines sit near zero. Dividing by those does not give a noisy answer, it gives a
meaningless one — the sign flips wherever the baseline is negative, and the magnitude
explodes wherever it is small. So:

- **`z`** — `(x − baseline) / baseline SD`, each trial in units of its own pre-stimulus
  variability. Defined for **every** trial of **every** source regardless of sign,
  dimensionless, and comparable across participants and sources. **This is the
  canonical form: use it for the ICs and for `ASSR-mask` alike.**
- **`rel`** — `x / baseline − 1`, the relative change, as a human-readable percentage.
  Written **only where the baseline is positive**; every other trial is `NaN`,
  deliberately, so a downstream mean cannot quietly average a sign-flipped value. That
  makes it complete for `ASSR-mask` and a *biased subset* on the IC rows, where roughly
  half the baselines are negative. Treat it as a readout for the mask, never as the
  input to a comparison that spans both kinds of source.

**Why one form for everything, rather than the best form for each row.** The point of
this extraction is to put the learned components and the fixed mask side by side and
ask which recovers the response better. Normalising them differently would confound
exactly that comparison: a difference between the two arms could then come from the
transform rather than from the spatial filter, and the two arms' effect sizes would not
even be in the same units. `z` is the form both rows can carry, so `z` is the one the
comparison runs on. `rel` stays in the file because a percentage change is easier to
report than a z-score — not because the analysis should switch between them.

Both are stored next to the raw power, so nothing is thrown away and the choice stays
reversible.

**This is not cosmetic — it changes the result.** Averaged as raw power, Placebo showed
a modest onset-locked rise while Psilocybin appeared to decline monotonically across the
whole epoch, which read as a slow non-stationarity swamping any response. That reading
was an artefact of the averaging: the trials differ in *level*, and with 9 trials the
group mean of raw power is dominated by those level differences rather than by the
stimulus-locked change inside each trial. Referencing every trial to its own baseline
first removes them, and **both** conditions then show a clear driven response peaking
inside the 0–0.5 s stimulus interval (Step 9). Treat any conclusion drawn from the
un-normalised group mean as superseded.

**What it still does not fix.** A drift *within* the 1.1 s epoch survives, because the
baseline only sets each trial's starting level. And normalising cannot recover
statistical power: 9 trials per participant is what the 12 s subset cache allows, and
the post-stimulus interval does not return to zero here, so the epochs are not fully
independent of one another.


In [ ]:
# The pre-stimulus interval each trial is referenced to.
baseline_mask = epoch_times < 0.0
print(
    f"Pre-stimulus interval: {int(baseline_mask.sum())} samples, "
    f"{epoch_times[0]:.3f} to {epoch_times[baseline_mask][-1]:.3f} s"
)

# ── Why two forms: how often is a per-trial baseline unusable as a divisor? ──
print("\nPer-trial baselines, by source (a ratio needs a POSITIVE baseline):")
print(f"  {'source':<10} {'condition':<11} {'negative':>10} {'near zero':>11} {'max |ratio|':>12}")
for s, source in enumerate(SOURCE_LABELS):
    for condition in CONDITIONS_TO_POOL:
        arr = trials[FREQ_SELECTIONS[0][0]][condition.value][:, s]  # (P, N, W)
        base = arr[..., baseline_mask].mean(axis=-1)
        # "Near zero" relative to the trial's own typical magnitude, which is what
        # decides whether the division explodes.
        near_zero = np.abs(base) < 0.1 * np.abs(arr).mean(axis=-1)
        print(
            f"  {source:<10} {condition.value:<11} "
            f"{f'{int((base < 0).sum())}/{base.size}':>10} "
            f"{f'{int(near_zero.sum())}/{base.size}':>11} "
            f"{np.abs(arr / base[..., None]).max():>12.3g}"
        )


def _normalise(array: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Per-trial baseline normalisation of a ``(P, S, N, W)`` array.

    Returns ``(rel, z, baseline_positive)``:

    * ``rel`` — ``x / baseline - 1``, NaN wherever the baseline is not positive. The
      NaN is the point: on a signed IVA source a negative baseline silently INVERTS
      the ratio, so those trials must not survive into a mean as numbers.
    * ``z`` — ``(x - baseline) / baseline SD``, defined for every trial whatever its
      sign, in units of that trial's own pre-stimulus variability.
    * ``baseline_positive`` — ``(P, S, N)`` mask of the trials ``rel`` is valid for.
    """
    base = array[..., baseline_mask].mean(axis=-1, keepdims=True)
    spread = array[..., baseline_mask].std(axis=-1, ddof=1, keepdims=True)

    positive = base > 0
    rel = np.where(positive, array / np.where(positive, base, 1.0) - 1.0, np.nan)

    # A zero-variance baseline would divide by zero; it never occurs on real wavelet
    # power, but guard rather than emit a silent inf.
    usable = spread > 0
    z = np.where(usable, (array - base) / np.where(usable, spread, 1.0), np.nan)
    return rel, z, positive[..., 0]


trials_rel: dict[str, dict[str, np.ndarray]] = {}
trials_z: dict[str, dict[str, np.ndarray]] = {}
baseline_positive: dict[str, dict[str, np.ndarray]] = {}

for name, _center, _halfwidth in FREQ_SELECTIONS:
    trials_rel[name], trials_z[name], baseline_positive[name] = {}, {}, {}
    for condition in CONDITIONS_TO_POOL:
        rel, z, positive = _normalise(trials[name][condition.value])
        trials_rel[name][condition.value] = rel
        trials_z[name][condition.value] = z
        baseline_positive[name][condition.value] = positive

print("\nNormalised arrays, same (participants x sources x trials x samples) layout:")
for name, _center, _halfwidth in FREQ_SELECTIONS:
    for condition in CONDITIONS_TO_POOL:
        rel = trials_rel[name][condition.value]
        valid = baseline_positive[name][condition.value]
        print(
            f"  {name:<9} {condition.value:<11} rel {rel.shape}  "
            f"{int(valid.sum())}/{valid.size} trial(s) with a positive baseline "
            f"({100 * valid.mean():.0f}%), the rest NaN in `rel`; `z` is complete"
        )

# Per source, so it is obvious which rows `rel` can actually be used on.
print("\nShare of trials with a usable (positive) baseline, per source:")
for s, source in enumerate(SOURCE_LABELS):
    shares = {
        condition.value: baseline_positive[FREQ_SELECTIONS[0][0]][condition.value][
            :, s
        ].mean()
        for condition in CONDITIONS_TO_POOL
    }
    verdict = (
        "rel usable"
        if min(shares.values()) > 0.99
        else "USE z — rel is NaN on much of this row"
    )
    joined = "  ".join(f"{c} {v:5.0%}" for c, v in shares.items())
    print(f"  {source:<10} {joined}   {verdict}")

---
## Step 9 — Write the extracted trials, and check they look like a response

One `.npz` per frequency selection, each **self-contained**: the trial arrays plus
every axis needed to interpret them — participant labels, source labels, the epoch time
base, the kept onsets, the frequency bins that were averaged, and the provenance of the
filters. Nothing downstream should have to re-derive a row's meaning from this
notebook's variable names.

Per-condition arrays are keyed `trials__<Condition>` rather than stacked into one
array, because the two conditions are free to keep a different number of trials and
padding them to a common length would invent data.

```python
with np.load(path, allow_pickle=True) as f:
    raw = f["trials__Placebo"]          # (participants, sources, trials, samples)
    rel = f["trials_rel__Placebo"]      # x / baseline - 1, NaN where baseline <= 0
    z   = f["trials_z__Placebo"]        # (x - baseline) / baseline SD, always defined
    labels = f["source_labels"]         # "IC 1" .. "IC K", "ASSR-mask"
    t      = f["times"]                 # seconds, 0 at onset
```

Each file carries the raw power **and** both per-trial normalisations from Step 8, so the choice of reference stays reversible.

The final output is a **sanity check on the extraction, not the analysis**: for each
source, the group-mean power in the driven interval against the pre-onset baseline,
plus the `ASSR-mask` time course printed sample by sample. What is being checked is
that a stimulus-locked change exists at all and sits in the right place — rising after
onset and falling back around the 0.5 s stimulus offset. If it does not, the epochs are
mis-timed or the rows are mis-indexed, and no downstream contrast is worth running. No
condition difference is tested here.

**Why a relative change and not a ratio.** The `ASSR-mask` row is a *non-negative*
average over electrodes, so a ratio would be fine for it. An IVA source is not: the
filter is a **signed** combination of channels, so a component's "power" can be
negative, and the ratio of two such numbers is uninterpretable — arbitrarily large,
negative, or near-infinite purely from a baseline close to zero. `(stimulus −
baseline) / |baseline|` is well defined for both kinds of row, so both can sit in one
table. Read the mask row first regardless: it is fixed by the montage and the paradigm,
so it is the assumption-free reference the learned rows are judged against.


In [ ]:
# ── Write one self-contained file per frequency selection ─────
# Provenance shared by every file: enough to tell which decomposition these filters
# came from and what was (not) done to the signal they were applied to.
zscore_note = zscore_mode.item() if zscore_mode is not None else "unrecorded"
trial_metadata = {
    "store_path": str(results.path),
    "variant": results.variant.value,
    "store_condition": results.condition.value,
    "music_type": MUSIC_TYPE.value,
    "n_pca": str(N_COMPONENTS_PCA),
    "filter_zscore_mode": zscore_note,
    "signal_zscore": "none — trials are raw wavelet power from the subset cache",
    "baseline_correction": "none",
    "units": "wavelet power",
    "binary_filter": "mean" if ASSR_MASK_NORMALIZE else "sum",
    "cache_n_times": str(raw_tracks[CONDITIONS_TO_POOL[0].value].shape[-1]),
    "sign_alignment": ALIGNMENT_NOTE,
}

trial_paths = []
if SAVE_TRIALS:
    TRIALS_DIR.mkdir(parents=True, exist_ok=True)
for name, _center, _halfwidth in FREQ_SELECTIONS:
    payload = {
        "participants": np.asarray(projected_participants, dtype=object),
        "source_labels": np.asarray(SOURCE_LABELS, dtype=object),
        "conditions": np.asarray([c.value for c in CONDITIONS_TO_POOL], dtype=object),
        "times": epoch_times,
        "sfreq": np.asarray(sfreq),
        "freqs": raw_freqs[selection_bins[name]],
        "selection": np.asarray(name),
        "epoch_pre": np.asarray(EPOCH_PRE),
        "epoch_post": np.asarray(EPOCH_POST),
        "binary_channels": np.asarray(assr_channels, dtype=object),
        "channel_names": np.asarray(list(results.channel_names), dtype=object),
    }
    for condition in CONDITIONS_TO_POOL:
        payload[f"trials__{condition.value}"] = trials[name][condition.value]
        # Both per-trial normalisations from Step 8, so nothing is thrown away.
        payload[f"trials_rel__{condition.value}"] = trials_rel[name][condition.value]
        payload[f"trials_z__{condition.value}"] = trials_z[name][condition.value]
        payload[f"baseline_positive__{condition.value}"] = baseline_positive[name][
            condition.value
        ]
        payload[f"onsets__{condition.value}"] = trial_onsets[condition.value]
    for key, value in trial_metadata.items():
        payload[f"meta__{key}"] = np.asarray(str(value))

    path = TRIALS_DIR / (
        f"assr_trials__{VARIANT.value}__{MUSIC_TYPE.value}__{name}"
        f"__pca{N_COMPONENTS_PCA}.npz"
    )
    if SAVE_TRIALS:
        np.savez_compressed(path, **payload)
        print(f"saved {path}  ({path.stat().st_size / 1e6:.2f} MB)")
    trial_paths.append(path)
if not SAVE_TRIALS:
    print("SAVE_TRIALS is False — nothing written; the arrays live in `trials`.")

# ── QC: does the driven interval stand out from the baseline? ─────────
# Computed on the PER-TRIAL normalised arrays from Step 8, not on raw power. Averaging
# raw power over trials is dominated by between-trial level differences, which is
# exactly what the normalisation removes — a QC table built on it would contradict the
# time course printed below. baseline_mask and stimulus_mask come from Steps 7-8.
records = []
for name, _center, _halfwidth in FREQ_SELECTIONS:
    for condition in CONDITIONS_TO_POOL:
        rel = trials_rel[name][condition.value]  # (P, S, N, W)
        z = trials_z[name][condition.value]
        for s, source in enumerate(SOURCE_LABELS):
            records.append(
                {
                    "selection": name,
                    "condition": condition.value,
                    "source": source,
                    # nanmean: `rel` is NaN wherever the baseline was not positive.
                    "rel": np.nanmean(rel[:, s][..., stimulus_mask]),
                    "z": np.nanmean(z[:, s][..., stimulus_mask]),
                }
            )
summary = pd.DataFrame.from_records(records)
n_trials = trial_onsets[CONDITIONS_TO_POOL[0].value].size
print(
    f"\nStimulus-interval mean of the per-trial normalised trials, "
    f"{len(projected_participants)} participants x {n_trials} trials — QC of the "
    f"extraction, not a test:"
)
display(
    summary.pivot_table(
        index=["selection", "source"], columns="condition", values=["rel", "z"]
    ).round(3)
)
print(
    "  Read the `z` block: it is the one form every source carries, so it is what a\n"
    "  mask-vs-IC comparison must run on. `rel` on an IC row is a nanmean over only\n"
    "  its positive-baseline trials — a biased subset — so it is a readout for\n"
    "  ASSR-mask only, never the input to a comparison spanning both."
)

# The shape over the epoch, now on the PER-TRIAL normalised data. A driven 40 Hz
# response should rise after onset and fall back around the stimulus offset; a
# monotonic drift across the whole window is a slow non-stationarity of the recording
# that per-trial baselining re-references to zero but does not remove.
mask_row = SOURCE_LABELS.index(BINARY_FILTER_LABEL)
marks = np.linspace(0, epoch_times.size - 1, 23).astype(int)
selection = FREQ_SELECTIONS[0][0]
print(
    f"\nPer-trial normalised {BINARY_FILTER_LABEL} at "
    f"{raw_freqs[selection_bins[selection]].tolist()} Hz, group+trial mean "
    "(0 = its own pre-stimulus level):"
)
print("  t (s)      : " + " ".join(f"{epoch_times[i]:6.2f}" for i in marks))
for condition in CONDITIONS_TO_POOL:
    rel = np.nanmean(trials_rel[selection][condition.value][:, mask_row], axis=(0, 1))
    print(
        f"  {condition.value[:6]:<6} rel: "
        + " ".join(f"{rel[i]:6.2f}" for i in marks)
    )
for condition in CONDITIONS_TO_POOL:
    z = np.nanmean(trials_z[selection][condition.value][:, mask_row], axis=(0, 1))
    print(
        f"  {condition.value[:6]:<6} z  : " + " ".join(f"{z[i]:6.2f}" for i in marks)
    )
print(f"  stimulus interval: 0.00-{AssrEpoch.STIMULUS_DURATION_S:.2f} s")

---
## Step 10 — Participant-level tests: condition contrast, and mask vs IC

The three collapses, then the statistics. `trials_z` arrives as
`(participants, sources, trials, samples)` and is reduced twice before anything is
tested:

```
(12, 6, 9, 275)  --time-->  (12, 6, 9)  --median over trials-->  (12, 6)
```

and the **12 participants are the unit of every test below**. Not the 108 trials: a
participant's 9 trials are correlated (measured ICC 0.08–0.14 for the mask), so a test
that treats them as 108 independent observations is anticonservative — on this data it
turns *p* = 0.375 into *p* = 0.089. Aggregating first is not a loss of power either;
for a balanced design the paired test on participant summaries and a random-intercept
mixed model give the **same** standard error, because `Var = σ_b²/P + σ_w²/(P·m)` and
the degrees of freedom are set by the number of participants either way.

**Polarity is re-anchored to the ASSR electrodes first.** A component's sign is fixed
by the run's PC1 alignment, which is not the same as "positive means more fronto-central
power" — measured here, the sign of a component's forward-pattern weight over the 35
mask electrodes splits across participants on every IC (8/12, 10/12, 9/12, 6/12, 5/12).
Left alone, a positive value would mean *more* power over that area for some
participants and *less* for others, and no direction could be predicted. So each
participant's component is flipped by

```
flip[p, k] = sign( mean forward-pattern weight of IC k over the ASSR electrodes )
```

after which a higher value means more 40 Hz power over that area for everyone. Every
component projects substantially onto those electrodes (median |w_mask| / |w_all| =
0.45–1.03), so the anchor is well determined rather than a coin flip on noise.

**Why this anchor and not one taken from the data.** The flip must not treat the two
conditions asymmetrically, because a paired test rests on them being exchangeable under
the null. The pattern is safe: this variant estimates **one** mixing matrix per
participant over the concatenated recording, so it is shared by both conditions and
structurally cannot favour either. An anchor read off the tested quantity is not —
"flip so Placebo is positive" forces the flipped Placebo value to be |v| ≥ 0 by
construction while leaving Psilocybin centred, manufacturing a positive difference under
the null. Simulated, that inflates the one-sided false-positive rate from 0.05 to
**0.68**. (`sign(v_placebo + v_psilocybin)` would be symmetric and therefore valid, but
it aligns to whichever way the noise points and means nothing physically.)

The flip is applied identically to both conditions, so the paired difference is
untouched and a genuine reversal between them survives — unlike taking |value|, which
would map `+2` against `−2` to a difference of zero.

**Exact Wilcoxon signed-rank.** The design is paired, so under the null a participant's
difference is equally likely to carry either sign. Wilcoxon enumerates all 2¹² = 4096
sign assignments over the **ranks** of |d|, making the *p*-value exact and free of any
distributional assumption. Ranking is why it is preferred here over the same
enumeration on the raw differences: it bounds how much any one participant can move the
result, which matters because the per-trial normalisation divides by a baseline SD
estimated from 25 samples and is heavy-tailed across trials (IQR/median 1.3–1.85). The
cost is that magnitude information is discarded.

**Know the floor before reading the output.** The enumeration bounds how small a *p*
can get: 2/4096 ≈ 0.00049 two-sided, 1/4096 ≈ 0.00024 one-sided, and only when all 12
participants point the same way. The design clears α = 0.05 comfortably; it cannot
produce a very small *p* however large the effect.

**Why not test |placebo − psilocybin|.** It would remove the need for a direction and
would also destroy the test: |d| ≥ 0 by construction, so "is it above zero?" is true
whenever there is any noise at all — simulated on null data it fires at **α = 1.00**.
The sign *is* the signal.

**Two families, both reported uncorrected.**

- **4b — Placebo vs Psilocybin**, one test per source.
- **4c — does an IC separate the conditions better than the mask?** One test per IC,
  **every IC, no selection**. Testing all five is what keeps this honest: picking the IC
  by its own effect size and then testing it against the mask on the same data would
  bias the result, because the IC was chosen for being large.

  The quantity is an **interaction**, not a within-condition comparison:

  ```
  d[p] = (placebo_ic − psilocybin_ic) − (placebo_mask − psilocybin_mask)
  ```

  i.e. each participant's condition contrast read through the component, minus the same
  contrast read through the fixed electrode selection. Zero means the component
  separates Placebo from Psilocybin exactly as well as the mask does; positive means
  better. That is the question the mask is here to answer — it is the reference
  *discriminator*, not merely another response detector.

  Comparing IC against mask **within** a condition instead would split this into two
  halves that only mean something together: their difference is algebraically this same
  interaction (verified to 9e-16), and each half on its own answers "which filter picks
  up more 40 Hz power", which is not what the conditions are being compared for.

  Two-sided, because nothing predicts that a learned component should beat a fixed
  selection. Both terms are in pre-stimulus SD units, so the subtraction is between
  comparable quantities.

**No multiplicity correction is applied**, by choice. 4b runs 6 tests and 4c runs 5 per
condition, so a *p* just under 0.05 on one row out of five is roughly what chance alone
delivers — read any single row accordingly, and weigh a result by whether the other
rows agree with it rather than by that row's *p* alone. The `effect` and `same sign`
columns are there for exactly that: 10 of 10 medians pointing one way is evidence of a
different kind from one row crossing a threshold.

**On the sign of an IC.** A component's polarity is fixed only up to the global sign
the run's PC1 alignment chose, so a negative value is not "less response" — the
component points the other way. Both families here test a *difference* (between
conditions, or against the mask), which is meaningful whatever the polarity, so this
does not affect them. It is why the "does a response exist" test is deliberately absent.


In [ ]:
# ── The two collapses: time, then trials ──────────────────────
if TEST_SELECTION not in trials_z:
    raise KeyError(
        f"TEST_SELECTION {TEST_SELECTION!r} is not among the extracted selections "
        f"{list(trials_z)}."
    )

if RESPONSE_MEASURE == "stimulus":
    def _reduce_time(z: np.ndarray) -> np.ndarray:
        """Mean over the driven interval."""
        return z[..., stimulus_mask].mean(axis=-1)
elif RESPONSE_MEASURE == "stimulus_minus_rest":
    def _reduce_time(z: np.ndarray) -> np.ndarray:
        """Driven interval minus the rest of the epoch, cancelling a residual offset."""
        return z[..., stimulus_mask].mean(axis=-1) - z[..., ~stimulus_mask].mean(axis=-1)
else:
    raise ValueError(
        f"RESPONSE_MEASURE must be 'stimulus' or 'stimulus_minus_rest'; got "
        f"{RESPONSE_MEASURE!r}."
    )

value = {}
for condition in CONDITIONS_TO_POOL:
    per_trial = _reduce_time(trials_z[TEST_SELECTION][condition.value])  # (P, S, N)
    value[condition.value] = np.median(per_trial, axis=2)  # (P, S)

n_test_participants = value[CONDITIONS_TO_POOL[0].value].shape[0]
n_test_trials = trials_z[TEST_SELECTION][CONDITIONS_TO_POOL[0].value].shape[2]
print(
    f"Response      : {RESPONSE_MEASURE} on {TEST_SELECTION} "
    f"({raw_freqs[selection_bins[TEST_SELECTION]].tolist()} Hz)"
)
print(
    f"Collapsed     : (P, S, N, W) -> (P, S) = "
    f"{value[CONDITIONS_TO_POOL[0].value].shape}, median over "
    f"{n_test_trials} trial(s)"
)


# ── Re-anchor each participant's component polarity to the ASSR area ──
# The sign of a component's forward-pattern weight over the mask electrodes is the
# mapping from "IC value" to "40 Hz power there". It is a property of the pattern, which
# this variant estimates ONCE per participant across both conditions, so flipping by it
# is symmetric in the conditions and cannot bias the paired contrast.
pattern_weight = channel_patterns[:, :, assr_mask].mean(axis=2)  # (P, K)
pattern_scale = np.abs(channel_patterns).mean(axis=2)  # (P, K) overall pattern size

flip = np.ones((n_test_participants, n_sources))
if ALIGN_POLARITY_TO_MASK:
    # The mask row needs no flip: it is a non-negative average over those very
    # electrodes, so "higher = more power there" already holds.
    flip[:, :n_components] = np.where(pattern_weight >= 0, 1.0, -1.0)

value = {c: flip * v for c, v in value.items()}

print("\nPolarity anchor — pattern weight over the ASSR electrodes:")
print(f"  {'IC':<6} {'flipped':>9} {'median |w_mask|/|w_all|':>25} {'anchor':>12}")
for k in range(n_components):
    ratio = np.abs(pattern_weight[:, k]) / pattern_scale[:, k]
    strong = "well determined" if np.median(ratio) > 0.2 else "WEAK"
    print(
        f"  IC {k + 1:<3} {int((flip[:, k] < 0).sum()):>6}/{n_test_participants} "
        f"{np.median(ratio):>25.3f} {strong:>12}"
    )
if ALIGN_POLARITY_TO_MASK:
    print(
        "  -> a higher value now means MORE 40 Hz power over the ASSR electrodes for\n"
        "     every participant, which is what makes the one-sided 4b meaningful."
    )
else:
    print("  -> ALIGN_POLARITY_TO_MASK is False; 4b's direction is NOT interpretable.")


# ── Exact Wilcoxon signed-rank test ───────────────────────────
def paired_test(differences: np.ndarray, alternative: str) -> dict:
    """Exact Wilcoxon signed-rank test on per-participant differences.

    The design is paired, so under the null a participant's difference is equally
    likely to carry either sign; Wilcoxon enumerates all 2**P sign assignments over the
    RANKS of |d|. Ranking bounds how far a single participant can move the result,
    which matters because the per-trial normalisation's denominator is heavy-tailed.
    ``method="exact"`` refuses the normal approximation, so the p-value is exact here.

    Deliberately NOT done: testing ``abs(d)``. It would remove the need for a direction
    and would also destroy the test — ``abs(d) >= 0`` by construction, so it fires
    whenever there is any noise at all (alpha = 1.00 on simulated null data).

    :param differences: One value per participant.
    :param alternative: ``"two-sided"``, or ``"greater"`` where the direction was fixed
        in advance and the polarity anchoring makes it meaningful.
    :return: Median difference, robust effect size (median over its own MAD), how many
        participants point positive, the alternative used, and the p-value.
    """
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d)]
    n = d.size
    if n < 2:
        raise ValueError(f"Need at least 2 participants; got {n}.")

    result = wilcoxon(d, alternative=alternative, method="exact")
    spread = 1.4826 * np.median(np.abs(d - np.median(d)))
    return {
        "median": float(np.median(d)),
        "effect": float(np.median(d) / spread) if spread > 0 else np.nan,
        "same sign": f"{int((d > 0).sum())}/{n}",
        "alt": alternative,
        "p": float(result.pvalue),
    }


floor = 2.0 / 2**n_test_participants
print(
    f"\nTest          : exact Wilcoxon signed-rank over {n_test_participants} "
    f"participants ({2**n_test_participants} sign assignments on the ranks)"
)
print(
    f"                smallest attainable p = {floor:.5f} two-sided, "
    f"{floor / 2:.5f} one-sided"
)

mask_index = SOURCE_LABELS.index(BINARY_FILTER_LABEL)

# ── 4b — Placebo vs Psilocybin, one test per source ───────────
rows = []
for s, source in enumerate(SOURCE_LABELS):
    d = value[CONDITIONS_TO_POOL[0].value][:, s] - value[CONDITIONS_TO_POOL[1].value][:, s]
    rows.append({"source": source, **paired_test(d, CONTRAST_ALTERNATIVE)})
contrast = pd.DataFrame(rows)

print(
    f"\n4b — {CONDITIONS_TO_POOL[0].value} minus {CONDITIONS_TO_POOL[1].value}, "
    f"paired over {n_test_participants} participants:"
)
display(contrast.set_index("source").round({"median": 4, "effect": 3, "p": 5}))
print(
    "  Uncorrected p over 6 tests — read a single row alongside `effect` and "
    "`same sign`,\n  not on its p alone."
)

# ── 4c — does an IC separate the conditions better than the mask? ─
# The INTERACTION, one test per IC: each participant's condition contrast through the
# component, minus the same contrast through the fixed electrode selection. Zero means
# the component discriminates exactly as well as the mask. Comparing the two filters
# within a condition instead would split this into two halves whose difference is this
# very quantity, and neither half alone speaks to condition discrimination.
mask_contrast = (
    value[CONDITIONS_TO_POOL[0].value][:, mask_index]
    - value[CONDITIONS_TO_POOL[1].value][:, mask_index]
)

rows = []
for s, source in enumerate(SOURCE_LABELS):
    if s == mask_index:
        continue
    ic_contrast = (
        value[CONDITIONS_TO_POOL[0].value][:, s]
        - value[CONDITIONS_TO_POOL[1].value][:, s]
    )
    # Two-sided: nothing predicts that a learned component should beat a fixed one.
    rows.append(
        {
            "source": source,
            "IC contrast": float(np.median(ic_contrast)),
            **paired_test(ic_contrast - mask_contrast, "two-sided"),
        }
    )
versus_mask = pd.DataFrame(rows)

print(
    f"\n4c — does an IC separate the conditions better than {BINARY_FILTER_LABEL}?"
    f"\n     (IC contrast) - (mask contrast), all 5 tested; 0 = as good as the mask."
    f"\n     Mask's own contrast: median {np.median(mask_contrast):+.4f}, "
    f"{int((mask_contrast > 0).sum())}/{n_test_participants} participants positive."
)
display(versus_mask.set_index("source").round({"IC contrast": 4, "median": 4, "effect": 3, "p": 5}))
print(
    "  Uncorrected p over 5 tests. Every IC is tested — none is selected by its own\n"
    "  effect size, which is what would otherwise bias this comparison."
)
better = int((versus_mask["median"] > 0).sum())
print(
    f"  Direction: {better}/{len(versus_mask)} IC(s) show a larger condition contrast "
    f"than the mask."
)
if floor > 0.01:
    print(
        f"\n  NOTE: with {n_test_participants} participants no p can fall below "
        f"{floor:.4f}, so a null result\n  here is a statement about the design as "
        "much as about the effect."
    )

---
## Step 11 — Summary figures

Two figures, answering the two questions the tests were built for.

**Figure 1 — the response over time, per source.** One panel per spatial filter, both
conditions overlaid. The line is the mean across the 12 participants of their own
median-over-trials time course; the band is the spread **across participants**, never
across trials — trials within a participant are correlated (ICC 0.08–0.14), so a band
drawn from them would look tight while saying nothing about how well the effect
generalises to a new person. The stimulus interval is shaded, and each panel carries its
one-sided *p* from 4b so the picture and the test are never read apart.

The IC panels are drawn **polarity-aligned**, the same flip the tests use: without it a
participant whose component loads negatively on the fronto-central electrodes would
cancel one who loads positively, and the group mean would collapse toward zero for
reasons that have nothing to do with the response.

**Figure 2 — the *p*-value summary**, which is the figure to read first.

- **Left: is Psilocybin lower than Placebo?** One row per source, showing the median
  paired difference `Placebo − Psilocybin` with a bootstrap interval. Positive is the
  predicted direction, so the *p* is one-sided `greater`. The `ASSR-mask` row is the
  fixed-electrode reference the components are judged against.
- **Right: does any component beat that reference?** `IC − ASSR-mask` within each
  condition, two-sided because nothing predicts which way this should go.

Intervals are a percentile bootstrap over participants and are shown to convey spread —
the *p*-values come from the exact Wilcoxon test in Step 10, not from the bootstrap, and
the two can disagree slightly at n = 12. Nothing here is corrected for multiplicity, so
read a marker against its neighbours as much as against α.


In [ ]:
# ── Per-participant time courses, polarity-aligned ────────────
# Collapse trials only: (P, S, N, W) -> (P, S, W). The flip is the same one the tests
# use, so the figure and the statistics describe the same quantity.
course = {
    condition.value: np.median(trials_z[TEST_SELECTION][condition.value], axis=2)
    * flip[:, :, None]
    for condition in CONDITIONS_TO_POOL
}


def _band(values: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Centre line and spread across the PARTICIPANT axis of a ``(P, W)`` array."""
    if COURSE_SPREAD == "sem":
        centre = values.mean(axis=0)
        half = values.std(axis=0, ddof=1) / np.sqrt(values.shape[0])
        return centre, centre - half, centre + half
    if COURSE_SPREAD == "iqr":
        return (
            np.median(values, axis=0),
            np.percentile(values, 25, axis=0),
            np.percentile(values, 75, axis=0),
        )
    raise ValueError(f"COURSE_SPREAD must be 'sem' or 'iqr'; got {COURSE_SPREAD!r}.")


contrast_by_source = contrast.set_index("source")

fig, axes = plt.subplots(2, 3, figsize=(15.5, 7.4), sharex=True, sharey=True)
for s, source in enumerate(SOURCE_LABELS):
    ax = axes.flat[s]
    ax.axvspan(
        0.0, AssrEpoch.STIMULUS_DURATION_S, color="0.55", alpha=0.11, lw=0, zorder=0
    )
    ax.axhline(0.0, color="0.45", lw=0.8, zorder=1)
    ax.axvline(0.0, color="0.35", lw=0.9, ls="--", zorder=1)
    for condition in CONDITIONS_TO_POOL:
        colour = CONDITION_COLORS[condition.value]
        centre, low, high = _band(course[condition.value][:, s])
        ax.fill_between(epoch_times, low, high, color=colour, alpha=0.18, lw=0, zorder=2)
        ax.plot(epoch_times, centre, color=colour, lw=1.9, zorder=3, label=condition.value)

    is_reference = source == BINARY_FILTER_LABEL
    row = contrast_by_source.loc[source]
    ax.set_title(
        f"{source}{'  (reference)' if is_reference else ''}",
        fontsize=12,
        fontweight="bold" if is_reference else "normal",
        loc="left",
    )
    # The test result belongs on the picture it describes.
    ax.text(
        0.985,
        0.955,
        f"Placebo > Psilocybin\np = {row['p']:.3f}   {row['same sign']}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=9,
        family="monospace",
        color="0.25" if row["p"] > ALPHA else "black",
        bbox=dict(
            boxstyle="round,pad=0.32",
            facecolor="white",
            edgecolor="0.75" if row["p"] > ALPHA else "black",
            alpha=0.88,
            lw=1.2 if row["p"] <= ALPHA else 0.8,
        ),
    )
    if s // 3 == 1:
        ax.set_xlabel("Time from stimulus onset (s)")
    if s % 3 == 0:
        ax.set_ylabel(f"40 Hz power\n(pre-stimulus SD)")

axes.flat[0].legend(loc="lower right", frameon=True, fontsize=10)
spread_label = (
    "mean +/- SEM across participants"
    if COURSE_SPREAD == "sem"
    else "median with 25-75 band across participants"
)
fig.suptitle(
    f"40 Hz response per spatial filter — {spread_label}, "
    f"{len(projected_participants)} participants x "
    f"{trial_onsets[CONDITIONS_TO_POOL[0].value].size} trials "
    f"({TEST_SELECTION}, polarity-aligned to the ASSR electrodes)",
    y=1.0,
    fontsize=12.5,
)
fig.tight_layout()
_save(fig, f"trial_course_by_source_{TEST_SELECTION}")
plt.show()


# ── Figure 2 — the p-value summary ────────────────────────────
def _bootstrap_ci(differences: np.ndarray) -> tuple[float, float]:
    """Percentile bootstrap interval for the median of *differences*, over participants.

    Shown to convey spread only. The p-values come from the exact Wilcoxon test, which
    does not use this; at n = 12 the two can disagree at the margin.
    """
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d)]
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    draws = np.median(d[rng.integers(0, d.size, size=(N_BOOTSTRAP, d.size))], axis=1)
    return tuple(np.percentile(draws, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)]))


fig, (ax_contrast, ax_versus) = plt.subplots(
    1, 2, figsize=(15.5, 5.0), gridspec_kw={"width_ratios": [1.0, 1.15]}
)

# --- Left: Placebo - Psilocybin, one row per source -----------
order = list(SOURCE_LABELS)
differences = {
    source: (
        value[CONDITIONS_TO_POOL[0].value][:, SOURCE_LABELS.index(source)]
        - value[CONDITIONS_TO_POOL[1].value][:, SOURCE_LABELS.index(source)]
    )
    for source in order
}
intervals = {source: _bootstrap_ci(d) for source, d in differences.items()}

# A robust x range. Scaling to the raw participant scatter lets one outlier — the
# per-trial normalisation divides by a 25-sample baseline SD, so they happen — stretch
# the axis and squash every interval into the middle. The range is set by the estimates
# and their intervals instead, and participants outside it are drawn as carets on the
# edge so none is silently dropped.
edges = [bound for pair in intervals.values() for bound in pair]
edges += [contrast_by_source.loc[source, "median"] for source in order]
edges += list(np.percentile(np.concatenate(list(differences.values())), [8, 92]))
span = max(edges) - min(edges)
x_lo, x_hi = min(edges) - 0.16 * span, max(edges) + 0.16 * span

for i, source in enumerate(order):
    y = len(order) - 1 - i
    d = differences[source]
    low, high = intervals[source]
    row = contrast_by_source.loc[source]
    significant = row["p"] <= ALPHA
    colour = "#1B5E20" if significant else "0.45"

    # Every participant, so the reader sees the n the test actually had.
    inside = (d >= x_lo) & (d <= x_hi)
    ax_contrast.scatter(
        d[inside], np.full(int(inside.sum()), y), s=13, color=colour, alpha=0.3,
        zorder=2, lw=0,
    )
    for value_off in d[~inside]:
        ax_contrast.scatter(
            [x_hi if value_off > x_hi else x_lo], [y], s=26, color=colour, alpha=0.55,
            marker=">" if value_off > x_hi else "<", zorder=2, lw=0,
        )

    ax_contrast.plot([low, high], [y, y], color=colour, lw=2.4, zorder=3, solid_capstyle="round")
    ax_contrast.scatter(
        [row["median"]], [y], s=95, color=colour, zorder=4,
        marker="D" if source == BINARY_FILTER_LABEL else "o",
        edgecolor="white", linewidth=1.1,
    )
    ax_contrast.text(
        1.005, y, f"p={row['p']:.3f}", transform=ax_contrast.get_yaxis_transform(),
        va="center", ha="left", fontsize=9.5, family="monospace",
        fontweight="bold" if significant else "normal", color=colour,
    )

n_off = sum(
    int(((d < x_lo) | (d > x_hi)).sum()) for d in differences.values()
)
if n_off:
    ax_contrast.text(
        0.5, -0.205,
        f"{n_off} participant point(s) beyond the axis, drawn as carets on the edge",
        transform=ax_contrast.transAxes, ha="center", va="top", fontsize=8.5,
        color="0.45", style="italic",
    )

ax_contrast.set_xlim(x_lo, x_hi)
ax_contrast.axvline(0.0, color="0.3", lw=1.0, zorder=1)
ax_contrast.set_yticks(range(len(order)))
ax_contrast.set_yticklabels(list(reversed(order)))
for tick, source in zip(ax_contrast.get_yticklabels(), reversed(order)):
    if source == BINARY_FILTER_LABEL:
        tick.set_fontweight("bold")
ax_contrast.set_xlabel("Placebo - Psilocybin  (pre-stimulus SD)")
ax_contrast.set_title(
    "Is Psilocybin lower than Placebo?\n"
    "one-sided, positive = predicted direction",
    loc="left", fontsize=12,
)

# --- Right: does an IC discriminate better than the reference? ---
# The interaction, so zero on this axis IS the ASSR mask: a marker to its right means
# the component separates the conditions more strongly than the fixed selection does.
ic_order = [s for s in SOURCE_LABELS if s != BINARY_FILTER_LABEL]
versus_indexed = versus_mask.set_index("source")
interaction = {
    source: (
        value[CONDITIONS_TO_POOL[0].value][:, SOURCE_LABELS.index(source)]
        - value[CONDITIONS_TO_POOL[1].value][:, SOURCE_LABELS.index(source)]
    )
    - mask_contrast
    for source in ic_order
}
inter_ci = {source: _bootstrap_ci(d) for source, d in interaction.items()}

v_edges = [bound for pair in inter_ci.values() for bound in pair]
v_edges += [versus_indexed.loc[source, "median"] for source in ic_order]
v_edges += list(np.percentile(np.concatenate(list(interaction.values())), [8, 92]))
v_span = max(v_edges) - min(v_edges)
v_lo, v_hi = min(v_edges) - 0.16 * v_span, max(v_edges) + 0.16 * v_span

for i, source in enumerate(ic_order):
    y = len(ic_order) - 1 - i
    d = interaction[source]
    low, high = inter_ci[source]
    row = versus_indexed.loc[source]
    significant = row["p"] <= ALPHA
    colour = "#1B5E20" if significant else "0.45"

    inside = (d >= v_lo) & (d <= v_hi)
    ax_versus.scatter(
        d[inside], np.full(int(inside.sum()), y), s=13, color=colour, alpha=0.3,
        zorder=2, lw=0,
    )
    for value_off in d[~inside]:
        ax_versus.scatter(
            [v_hi if value_off > v_hi else v_lo], [y], s=26, color=colour, alpha=0.55,
            marker=">" if value_off > v_hi else "<", zorder=2, lw=0,
        )
    ax_versus.plot([low, high], [y, y], color=colour, lw=2.4, zorder=3,
                   solid_capstyle="round")
    ax_versus.scatter(
        [row["median"]], [y], s=95, color=colour, zorder=4,
        edgecolor="white", linewidth=1.1,
    )
    ax_versus.text(
        1.005, y, f"p={row['p']:.3f}", transform=ax_versus.get_yaxis_transform(),
        va="center", ha="left", fontsize=9.5, family="monospace",
        fontweight="bold" if significant else "normal", color=colour,
    )

n_off_v = sum(int(((d < v_lo) | (d > v_hi)).sum()) for d in interaction.values())
if n_off_v:
    ax_versus.text(
        0.5, -0.205,
        f"{n_off_v} participant point(s) beyond the axis, drawn as carets on the edge",
        transform=ax_versus.transAxes, ha="center", va="top", fontsize=8.5,
        color="0.45", style="italic",
    )

ax_versus.set_xlim(v_lo, v_hi)
# The zero line IS the mask; the axis label and subtitle already say so, so it carries
# no annotation of its own — one there collides with the title.
ax_versus.axvline(0.0, color="0.3", lw=1.6, zorder=1)
ax_versus.set_yticks(range(len(ic_order)))
ax_versus.set_yticklabels(list(reversed(ic_order)))
ax_versus.set_xlabel(
    f"(IC contrast) - ({BINARY_FILTER_LABEL} contrast)   (pre-stimulus SD)"
)
ax_versus.set_title(
    f"Does any component separate the conditions better than {BINARY_FILTER_LABEL}?\n"
    "two-sided; 0 = exactly as good as the fixed electrode selection",
    loc="left", fontsize=12,
)

better = int((versus_mask["median"] > 0).sum())
fig.suptitle(
    f"Exact Wilcoxon signed-rank, n = {n_test_participants} participants "
    f"(floor p = {floor:.5f} two-sided) — uncorrected over "
    f"{len(contrast) + len(versus_mask)} tests; "
    f"{better}/{len(versus_mask)} IC(s) out-separate the reference",
    y=1.0, fontsize=12.5,
)
fig.tight_layout()
_save(fig, f"pvalue_summary_{TEST_SELECTION}")
plt.show()

print(
    f"Bars/bands: {spread_label}. Intervals on Figure 2 are a "
    f"{N_BOOTSTRAP:,}-draw percentile bootstrap over participants, shown for spread "
    f"only —\nthe p-values are the exact Wilcoxon ones from Step 10 and do not come "
    f"from it."
)